# Pi05 GPU Inference (PyTorch + CUDA)

Loads Eugene's finetuned Pi05 checkpoint and runs inference on RTX 3090.
Expected: ~1-5s per action chunk (vs ~90s on CPU with OpenVINO).

In [ ]:
import torch
import numpy as np
import cv2
from collections import deque
from matplotlib import pyplot as plt

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# --- Config ---
CHECKPOINT_PATH = "pi05/pi05_eugene.ckpt"
DEVICE = "cuda"  # "cuda" or "cpu"
TASK = "pick a can and place it in a bowl"

JOINT_NAMES = [
    "shoulder_pan",
    "shoulder_lift",
    "elbow_flex",
    "wrist_flex",
    "wrist_roll",
    "gripper",
]

# Camera config (SharedCamera UVC device indices)
OVERHEAD_CAMERA_INDEX = "/dev/v4l/by-id/usb-UGREEN_Camera_2K_UGREEN_Camera_2K_SN0001-video-index0"
ARM_CAMERA_INDEX = "/dev/v4l/by-id/usb-Innomaker_Innomaker-U20CAM-1080p-S1_SN0001-video-index0"
FLIP_CAMERAS = {"arm"}  # Cameras mounted upside down

CAMERA_WIDTH = 640
CAMERA_FPS = 30
CAMERA_HEIGHT = 480

## Load Model

In [ ]:
from physicalai.policies.pi05 import Pi05

# Load to CPU first to avoid OOM if GPU has residual allocations
policy = Pi05.load_from_checkpoint(CHECKPOINT_PATH, map_location="cpu", compile_model=False)
policy = policy.to(DEVICE)
policy.eval()
print(f"Model loaded on {DEVICE}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Connect Cameras & Robot

In [ ]:
from cv2_enumerate_cameras import enumerate_cameras

print("Available cameras:")
for c in enumerate_cameras(apiPreference=cv2.CAP_GSTREAMER):
    print(f" - {c.name} (index {c.index})")

In [ ]:
from physicalai.capture import SharedCamera
from lerobot.robots.so_follower import SO101Follower, SO101FollowerConfig

# Cameras via SharedCamera (UVC)
cameras = {
    "overhead": SharedCamera("uvc", device=OVERHEAD_CAMERA_INDEX, width=CAMERA_WIDTH, height=CAMERA_HEIGHT, fps=CAMERA_FPS),
    "arm": SharedCamera("uvc", device=ARM_CAMERA_INDEX, width=CAMERA_WIDTH, height=CAMERA_HEIGHT, fps=CAMERA_FPS),
}
for name, cam in cameras.items():
    cam.connect()
    print(f"Camera '{name}' connected: {cam.actual_width}x{cam.actual_height} @ {cam.actual_fps}fps")

# Robot without cameras
robot_cfg = SO101FollowerConfig(
    port="/dev/ttyACM1",
    id="my_so101_follower",

)

robot = SO101Follower(robot_cfg)
robot.connect()
print("Robot connected")

## Get Observation & Display

In [ ]:
def get_full_observation() -> dict:
    """Combine robot state + SharedCamera images into one dict."""
    obs = robot.get_observation()
    for cam_key, cam in cameras.items():
        frame = cam.read_latest()
        obs[cam_key] = frame.data  # numpy HWC RGB
    return obs

def viz_observation(obs: dict):
    print("Joint positions:")
    for jn in JOINT_NAMES:
        print(f"  {jn}: {obs[f'{jn}.pos']:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, cam_key in zip(axes, ["overhead", "arm"]):
        img = np.ascontiguousarray(obs[cam_key])
        if cam_key in FLIP_CAMERAS:
            img = cv2.rotate(img, cv2.ROTATE_180)
        ax.imshow(img)
        ax.set_title(f"{cam_key} ({img.shape})")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

obs = get_full_observation()
viz_observation(obs)

## Build Observation for Model

In [ ]:
from physicalai.data.observation import Observation

def build_observation(obs: dict) -> Observation:
    """Convert robot observation to Pi05 Observation."""
    # State: (1, 6) float32
    state = torch.tensor(
        [[obs[f"{jn}.pos"] for jn in JOINT_NAMES]], dtype=torch.float32, device=DEVICE
    )

    # Images: {name: (1, 3, H, W) float32 in [0, 1]}
    def img_to_tensor(img: np.ndarray) -> torch.Tensor:
        t = torch.from_numpy(img.copy()).float() / 255.0  # (H, W, 3) -> [0,1]
        t = t.permute(2, 0, 1).unsqueeze(0)  # (1, 3, H, W)
        return t.to(DEVICE)

    images = {}
    for cam_key in ["overhead", "arm"]:
        img = np.ascontiguousarray(obs[cam_key])
        if cam_key in FLIP_CAMERAS:
            img = cv2.rotate(img, cv2.ROTATE_180)
        images[cam_key] = img_to_tensor(img)

    return Observation(state=state, images=images, task=TASK)

observation = build_observation(get_full_observation())
print(f"State shape: {observation.state.shape}")
for k, v in observation.images.items():
    print(f"Image '{k}': {v.shape}, device={v.device}")

## Run Inference (single step, timed)

In [ ]:
import time

# Reset action queue to get a fresh prediction
policy._action_queue = deque()

# Warm up GPU
if DEVICE == "cuda":
    torch.cuda.synchronize()

t0 = time.perf_counter()
with torch.no_grad():
    action = policy.select_action(observation)
if DEVICE == "cuda":
    torch.cuda.synchronize()
latency = time.perf_counter() - t0

action_np = action.detach().cpu().numpy()
if action_np.ndim == 2:
    action_np = action_np[0]

print(f"Inference latency: {latency*1000:.1f} ms")
print(f"Action shape: {action_np.shape}")
print(f"Action: {action_np}")
print()
for i, jn in enumerate(JOINT_NAMES):
    print(f"  {jn}: {action_np[i]:.4f}")

## Benchmark (5 inferences)

In [ ]:
latencies = []
for i in range(5):
    policy._action_queue = deque()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        action = policy.select_action(observation)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    latencies.append(time.perf_counter() - t0)

print(f"Latencies: {[f'{l*1000:.1f}ms' for l in latencies]}")
print(f"Mean: {np.mean(latencies)*1000:.1f} ms")
print(f"Min:  {np.min(latencies)*1000:.1f} ms")
print(f"Max:  {np.max(latencies)*1000:.1f} ms")

## Send Action to Robot

**Run only when ready to move.**

In [ ]:
action_dict = {f"{jn}.pos": float(action_np[i]) for i, jn in enumerate(JOINT_NAMES)}
robot.send_action(action_dict)
print(f"Sent: {action_dict}")

## Rollout Loop (observe → infer → execute full chunk)

Runs a continuous loop: get observation, predict action chunk, execute all actions in the chunk at 30Hz, then repeat. **Interrupt the kernel to stop.**

In [ ]:
# FPS = 30
# CHUNK_SIZE = 50  # Pi05 default action chunk length

# try:
#     step = 0
#     while True:
#         # Get fresh observation
#         obs = get_full_observation()
#         observation = build_observation(obs)

#         # Generate full action chunk
#         policy._action_queue = deque()
#         t0 = time.perf_counter()
#         with torch.no_grad():
#             action = policy.select_action(observation)
#         infer_ms = (time.perf_counter() - t0) * 1000

#         # Drain remaining actions from the queue (select_action returns first, rest are queued)
#         actions = [action.detach().cpu().numpy()]
#         while len(policy._action_queue) > 0:
#             a = policy._action_queue.popleft()
#             actions.append(a.detach().cpu().numpy() if isinstance(a, torch.Tensor) else a)

#         print(f"[step {step}] Inferred {len(actions)} actions in {infer_ms:.0f}ms")

#         # Execute each action in the chunk at target FPS
#         for i, act in enumerate(actions):
#             t_act = time.perf_counter()
#             act = np.squeeze(act)
#             action_dict = {f"{jn}.pos": float(act[j]) for j, jn in enumerate(JOINT_NAMES)}
#             robot.send_action(action_dict)

#             # Sleep to maintain target FPS
#             elapsed = time.perf_counter() - t_act
#             sleep_time = (1.0 / FPS) - elapsed
#             if sleep_time > 0:
#                 time.sleep(sleep_time)

#         step += 1

# except KeyboardInterrupt:
#     print(f"\nStopped after {step} chunks")

## Async Threaded Rollout

Background thread runs GPU inference while the main thread sends actions at 30Hz.
Lerp-blends overlapping chunks for smooth transitions. **Interrupt the kernel to stop.**

In [ ]:
import threading

class QueueMixer:
    """Action queue with lerp blending between chunks."""

    def __init__(self, lerp_duration: int = 5):
        self.queue: np.ndarray | None = None
        self.lerp_duration = lerp_duration
        self.index = 0

    def add(self, chunk: np.ndarray, offset: int = 0):
        if self.queue is None or self.index >= len(self.queue):
            self.queue = chunk[offset:]
            self.index = 0
            return

        remaining = self.queue[self.index:]
        incoming = chunk[offset:]
        n_remain = len(remaining)
        lerp_dur = min(n_remain, self.lerp_duration)

        weights = np.maximum(1.0 - np.arange(n_remain) / max(lerp_dur, 1), 0.0)[:, np.newaxis]
        n_blend = min(n_remain, len(incoming))
        blended = weights[:n_blend] * remaining[:n_blend] + (1.0 - weights[:n_blend]) * incoming[:n_blend]
        self.queue = np.concatenate([blended, incoming[n_blend:]], axis=0).astype(np.float32)
        self.index = 0

    def pop(self) -> np.ndarray | None:
        if self.queue is None or self.index >= len(self.queue):
            return None
        action = self.queue[self.index]
        self.index += 1
        return action

    @property
    def remaining(self) -> int:
        if self.queue is None:
            return 0
        return max(len(self.queue) - self.index, 0)


class GpuInferenceThread:
    """Background thread that runs PyTorch GPU inference on demand."""

    def __init__(self, policy, device: str):
        self.policy = policy
        self.device = device
        self._lock = threading.Lock()
        self._obs_slot: dict | None = None
        self._obs_ready = threading.Event()
        self._running_inference = False
        self._request_time = 0.0
        self._result_lock = threading.Lock()
        self._result_slot: tuple[np.ndarray, float] | None = None
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, name="GpuInferenceThread", daemon=True)
        self.inference_count = 0

    def start(self):
        self._thread.start()

    def stop(self):
        self._stop.set()
        self._obs_ready.set()
        self._thread.join(timeout=10.0)

    def request(self, obs_dict: dict) -> bool:
        with self._lock:
            if self._obs_slot is not None or self._running_inference:
                return False
            self._obs_slot = obs_dict
            self._request_time = time.perf_counter()
        self._obs_ready.set()
        return True

    @property
    def busy(self) -> bool:
        with self._lock:
            return self._obs_slot is not None or self._running_inference

    @property
    def alive(self) -> bool:
        return self._thread.is_alive()

    @property
    def busy_duration(self) -> float:
        """Seconds since last request was submitted. 0 if not busy."""
        with self._lock:
            if not (self._obs_slot is not None or self._running_inference):
                return 0.0
            return time.perf_counter() - self._request_time

    def force_reset(self):
        """Clear stuck state so new requests can be submitted."""
        with self._lock:
            self._obs_slot = None
            self._running_inference = False
        print("[InferenceThread] Force reset — cleared stuck state")

    def get_result(self) -> tuple[np.ndarray, float] | None:
        with self._result_lock:
            r = self._result_slot
            self._result_slot = None
        return r

    def _run(self):
        while not self._stop.is_set():
            self._obs_ready.wait()
            self._obs_ready.clear()
            if self._stop.is_set():
                break

            with self._lock:
                obs_dict = self._obs_slot
                self._obs_slot = None
                self._running_inference = True

            if obs_dict is None:
                with self._lock:
                    self._running_inference = False
                continue

            self.inference_count += 1
            t0 = time.perf_counter()

            try:
                observation = build_observation(obs_dict)
                self.policy._action_queue = deque()

                with torch.no_grad():
                    action = self.policy.select_action(observation)
                if self.device == "cuda":
                    torch.cuda.synchronize()

                # Collect full chunk
                actions = [action.detach().cpu().numpy()]
                while len(self.policy._action_queue) > 0:
                    a = self.policy._action_queue.popleft()
                    actions.append(a.detach().cpu().numpy() if isinstance(a, torch.Tensor) else a)

                chunk = np.array([np.squeeze(a) for a in actions], dtype=np.float32)
            except Exception as e:
                import traceback
                print(f"[InferenceThread] Error: {e}")
                traceback.print_exc()
                with self._lock:
                    self._running_inference = False
                continue

            latency = time.perf_counter() - t0
            with self._result_lock:
                self._result_slot = (chunk, latency)
            with self._lock:
                self._running_inference = False

print("QueueMixer and GpuInferenceThread defined")

In [ ]:
ASYNC_FPS = 30
QUEUE_THRESHOLD_FRAC = 0.5  # Request new inference when queue drops below this fraction
LERP_DURATION = 5           # Frames over which to blend old/new chunks
PLOT_INTERVAL = 1           # Seconds between live plot updates

# --- Warm-up to determine chunk size ---
warmup_obs = get_full_observation()
warmup_observation = build_observation(warmup_obs)
policy._action_queue = deque()
with torch.no_grad():
    warmup_action = policy.select_action(warmup_observation)
warmup_actions = [warmup_action.detach().cpu().numpy()]
while len(policy._action_queue) > 0:
    a = policy._action_queue.popleft()
    warmup_actions.append(a.detach().cpu().numpy() if isinstance(a, torch.Tensor) else a)
warmup_chunk = np.array([np.squeeze(a) for a in warmup_actions], dtype=np.float32)
chunk_size = warmup_chunk.shape[0]
threshold = int(chunk_size * QUEUE_THRESHOLD_FRAC)
print(f"Chunk size: {chunk_size}, threshold: {threshold}")

# --- Telemetry collector ---
telemetry = {
    "t": [],                  # wall time per step
    "actions_sent": [],       # action_array for each action sent
    "queue_depth": [],        # (t, remaining) per step
    "infer_requests": [],     # t when inference was requested
    "infer_completions": [],  # (t, latency_ms, offset)
    "chunks_received": [],    # (t_received, chunk_array, inference_num)
    "holds": [],              # t when queue was empty (holding position)
}
t_start = time.perf_counter()

# --- Live plotting ---
import matplotlib.patches as mpatches
from IPython.display import display

def render_telemetry(telemetry, threshold, chunk_size):
    """Render telemetry to a matplotlib figure."""
    actions_arr = np.array(telemetry["actions_sent"])
    t_arr = np.array(telemetry["t"])
    if len(t_arr) < 2:
        return None

    queue_t, queue_d = zip(*telemetry["queue_depth"])
    n_joints = len(JOINT_NAMES)
    n_chunks = len(telemetry["chunks_received"])
    chunk_cmap = plt.cm.Set2

    fig, axes = plt.subplots(n_joints + 1, 1, figsize=(16, 2.5 * (n_joints + 1)), sharex=True)

    for j, jn in enumerate(JOINT_NAMES):
        ax = axes[j]
        ax.plot(t_arr, actions_arr[:, j], color="black", linewidth=1.2, zorder=3)

        for idx, (t_recv, chunk, _) in enumerate(telemetry["chunks_received"]):
            color = chunk_cmap(idx % 8)
            chunk_t = t_recv + np.arange(len(chunk)) / ASYNC_FPS
            ax.plot(chunk_t, chunk[:, j], color=color, alpha=0.4, linewidth=1.5, zorder=2)

        for t_req in telemetry["infer_requests"]:
            ax.axvline(t_req, color="green", alpha=0.3, linewidth=0.8, linestyle="--")
        for t_comp, _, _ in telemetry["infer_completions"]:
            ax.axvline(t_comp, color="blue", alpha=0.3, linewidth=0.8, linestyle="-.")
        for ht in telemetry["holds"]:
            ax.axvspan(ht, ht + 1/ASYNC_FPS, color="red", alpha=0.15)

        ax.set_ylabel(jn, fontsize=9)
        ax.tick_params(labelsize=8)

    ax_q = axes[-1]
    ax_q.fill_between(queue_t, queue_d, alpha=0.3, color="steelblue")
    ax_q.plot(queue_t, queue_d, color="steelblue", linewidth=1)
    ax_q.axhline(threshold, color="orange", linestyle="--", linewidth=1, label=f"threshold ({threshold})")
    for t_req in telemetry["infer_requests"]:
        ax_q.axvline(t_req, color="green", alpha=0.5, linewidth=1, linestyle="--")
    for t_comp, _, _ in telemetry["infer_completions"]:
        ax_q.axvline(t_comp, color="blue", alpha=0.5, linewidth=1, linestyle="-.")
    ax_q.set_ylabel("Queue", fontsize=9)
    ax_q.set_xlabel("Time (s)", fontsize=10)
    ax_q.legend(loc="upper right", fontsize=8)
    ax_q.tick_params(labelsize=8)

    legend_handles = [
        mpatches.Patch(color="black", label="Sent to robot"),
        mpatches.Patch(color="green", alpha=0.5, label="Infer requested"),
        mpatches.Patch(color="blue", alpha=0.5, label="Chunk received"),
        mpatches.Patch(color="red", alpha=0.3, label="Hold"),
    ]
    for idx in range(min(n_chunks, 6)):
        legend_handles.append(mpatches.Patch(color=chunk_cmap(idx % 8), alpha=0.5, label=f"Chunk #{idx+1}"))
    fig.legend(handles=legend_handles, loc="upper center", ncol=min(len(legend_handles), 5), fontsize=8, bbox_to_anchor=(0.5, 1.02))

    n_holds = len(telemetry["holds"])
    fig.suptitle(f"Live — {len(t_arr)} steps, {n_chunks} inferences, {n_holds} holds, t={t_arr[-1]:.1f}s", fontsize=11, y=1.04)
    plt.tight_layout()
    return fig

# Show initial empty plot placeholder
plot_handle = display(plt.figure(figsize=(16, 2)), display_id=True)
plt.close()

# --- Init ---
queue = QueueMixer(lerp_duration=LERP_DURATION)
queue.add(warmup_chunk)  # Seed so robot can move immediately

infer_thread = GpuInferenceThread(policy, DEVICE)
infer_thread.start()

goal_time = 1.0 / ASYNC_FPS
last_action = warmup_chunk[0]
step = 0
hold_count = 0
last_plot_time = 0.0

print(f"Async rollout at {ASYNC_FPS} Hz — interrupt kernel to stop")

try:
    while True:
        loop_start = time.perf_counter()
        t_rel = loop_start - t_start

        # 1. Check for inference result
        result = infer_thread.get_result()
        if result is not None:
            chunk, latency = result
            offset = int(latency * ASYNC_FPS)
            queue.add(chunk, offset=offset)
            queue.lerp_duration = max(offset, 1)
            telemetry["infer_completions"].append((t_rel, latency * 1000, offset))
            telemetry["chunks_received"].append((t_rel, chunk.copy(), infer_thread.inference_count))
            print(f"[t={t_rel:.1f}s] <<< inference #{infer_thread.inference_count}: "
                  f"{latency*1000:.0f}ms, offset={offset}, queue={queue.remaining}")

        # 2. Request new inference when queue is low (with stuck/dead detection)
        INFER_TIMEOUT = 15.0
        if queue.remaining <= threshold:
            if not infer_thread.alive:
                print(f"[t={t_rel:.1f}s] !!! Inference thread DEAD — restarting")
                infer_thread = GpuInferenceThread(policy, DEVICE)
                infer_thread.start()
            elif infer_thread.busy_duration > INFER_TIMEOUT:
                print(f"[t={t_rel:.1f}s] !!! Inference stuck for {infer_thread.busy_duration:.0f}s — force resetting")
                infer_thread.force_reset()

            if not infer_thread.busy:
                obs = get_full_observation()
                submitted = infer_thread.request(obs)
                if submitted:
                    telemetry["infer_requests"].append(t_rel)
                    print(f"[t={t_rel:.1f}s] >>> inference requested (queue={queue.remaining}/{chunk_size})")

        # 3. Pop action
        action = queue.pop()
        if action is not None:
            last_action = action
            hold_count = 0
        else:
            action = last_action
            hold_count += 1
            telemetry["holds"].append(t_rel)
            if hold_count == 1 or hold_count % 30 == 0:
                print(f"[t={t_rel:.1f}s] !!! Queue empty, holding (hold={hold_count})")

        # 4. Send to robot
        if action is not None:
            action_dict = {f"{jn}.pos": float(action[i]) for i, jn in enumerate(JOINT_NAMES)}
            robot.send_action(action_dict)

        # 5. Record telemetry
        telemetry["t"].append(t_rel)
        telemetry["actions_sent"].append(action.copy() if action is not None else last_action.copy())
        telemetry["queue_depth"].append((t_rel, queue.remaining))

        # 6. Live plot update (throttled)
        if t_rel - last_plot_time > PLOT_INTERVAL and len(telemetry["t"]) > 10:
            fig = render_telemetry(telemetry, threshold, chunk_size)
            if fig is not None:
                plot_handle.update(fig)
                plt.close(fig)
            last_plot_time = t_rel

        # 7. Maintain FPS
        elapsed = time.perf_counter() - loop_start
        sleep_time = goal_time - elapsed
        if sleep_time > 0:
            time.sleep(sleep_time)

        step += 1

except KeyboardInterrupt:
    print(f"\nStopped after {step} steps, {infer_thread.inference_count} inferences, "
          f"{len(telemetry['holds'])} holds, duration={time.perf_counter() - t_start:.1f}s")

infer_thread.stop()
print("Inference thread stopped")

# Final plot
fig = render_telemetry(telemetry, threshold, chunk_size)
if fig is not None:
    plot_handle.update(fig)
    plt.close(fig)

if telemetry["infer_completions"]:
    latencies_ms = [lat for _, lat, _ in telemetry["infer_completions"]]
    t_arr = np.array(telemetry["t"])
    print(f"\nInference latencies: mean={np.mean(latencies_ms):.0f}ms, "
          f"min={np.min(latencies_ms):.0f}ms, max={np.max(latencies_ms):.0f}ms")
    print(f"Queue empty fraction: {len(telemetry['holds'])/max(len(t_arr),1)*100:.1f}%")
    print(f"Total duration: {t_arr[-1]:.1f}s, steps: {len(t_arr)}, effective Hz: {len(t_arr)/t_arr[-1]:.1f}")

## Disconnect

In [ ]:
for name, cam in cameras.items():
    cam.disconnect()

    print(f"Camera '{name}' disconnected")
    
robot.disconnect()
print("Robot disconnected")